# Infaction Site Grouping — executable Proximity checkpoint companion
This notebook extends the existing **locked v0.4.6 executable** Faction→Zone→Location→Placement→City pipeline with a **non-mutating Site grouping stage** using the original Book1 118×118 Proximity matrix and the documented Site checkpoint algorithm.

**Status:** Site module is experimental, NOT a verified implementation of the current **v0.4.7 directional-Containment specification**. The v0.4.7 71-Location selection/repair and derived directional Containment routing are not fully integrated into the old v0.4.6 executable notebook. Do not label these results production v0.4.7.

No original Book1, placement, City, or Site design checkpoints are changed.


In [ ]:
import urllib.request,json,contextlib,io,types
ROOT="https://raw.githubusercontent.com/playrenathea/rantau/main/"
def get_text(path):return urllib.request.urlopen(ROOT+path,timeout=60).read().decode("utf-8")
base=json.loads(get_text("notebooks/Infaction_Locked_City_Placement_v0.4.6.ipynb"))
scope={"__name__":"__main__"}
for idx in (1,2,3):
    exec(compile("".join(base["cells"][idx]["source"]),f"base_cell_{idx}","exec"),scope)
print("Canonical old executable selection/placement/City preflight passed")


In [ ]:
from pathlib import Path
site_mod=types.ModuleType("infaction_site_proximity_checkpoint")
exec(compile(get_text("infaction_site_proximity_checkpoint.py"),"infaction_site_proximity_checkpoint.py","exec"),site_mod.__dict__)
master=scope["MASTER"]
scales={name:int(row["scale"]) for name,row in master["raw"]["place_catalog"].items()}
effective={**scales,**{"Fortress":3,"Palace":3,"Mine":3,"Factory":3,"Town Hall":2}}
site_candidates=[p for p in scope["place_names"] if effective.get(p) in (2,3,4)]
assert len(site_candidates)==50
site_result=site_mod.group_sites(scope["placed"],scope["result"],scope["SITE_AFFINITY"],site_candidates,scope["city_scores"],seed=scope["SEED"])
assert site_result["grouped_location_count"]+site_result["ungrouped_location_count"]==len(scope["placed"]["locations"])
print("Seed",scope["SEED"],"Sites",len(site_result["sites"]),"grouped",site_result["grouped_location_count"])
for s in site_result["sites"]:print(s["name"],"in",s["parent_zone"] or "Standalone",":",", ".join(s["children"]))
Path(f'infaction_world_with_sites_experimental_{scope["SEED"]}.json').write_text(json.dumps({"world":scope["world"],"placement":scope["placed"],"city":scope["result"],"site_grouping":site_result},indent=2,ensure_ascii=False))
